# Day 4: Manual Sentiment Labelling

This notebook prepares the labelling file.

The frozen dataset contains 111 headlines. The target final labels are `positive`, `neutral`, or `negative`. `review` is used temporarily for an uncertain row, but every `review` row will be resolved before model training.

## Draft labelling rubric


- **Positive:** The headline describes falling prices, improved affordability, increased supply, relief, or another clearly favourable development.
- **Negative:** The headline describes rising prices, inflation pressure, scarcity, worsening hardship, or unaffordability.
- **Neutral:** The headline reports information without a clear positive or negative tone.
- **Review:** The headline is ambiguous, mixed, peripheral, or difficult to classify. Resolve it before training.

Label the headline's expressed tone, not whether the event is objectively good or bad. Record a short reason and confidence for every label.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

CLEAN_DIR = PROJECT_DIR / 'data' / 'clean'
FROZEN_PATH = CLEAN_DIR / 'articles_frozen.csv'
LABEL_PATH = CLEAN_DIR / 'articles_to_label.csv'
LABELED_PATH = CLEAN_DIR / 'articles_labeled.csv'

frozen = pd.read_csv(FROZEN_PATH)
print('Frozen rows:', len(frozen))
print('Frozen columns:', list(frozen.columns))

Frozen rows: 111
Frozen columns: ['query', 'feed_url', 'article_url', 'headline', 'headline_raw', 'pub_date_raw', 'source_label', 'source_domain', 'pub_datetime', 'collection_time', 'headline_key', 'has_food_term', 'has_price_term', 'has_topic_phrase', 'is_topic_relevant']


In [2]:
if LABEL_PATH.exists():
    labeling = pd.read_csv(LABEL_PATH)
    print('Existing labelling file loaded:', LABEL_PATH)
else:
    labeling = frozen[['article_url', 'headline', 'pub_datetime', 'source_label']].copy()
    labeling.insert(0, 'row_id', range(1, len(labeling) + 1))
    labeling['sentiment_label'] = ''
    labeling['label_reason'] = ''
    labeling['confidence'] = ''
    labeling.to_csv(LABEL_PATH, index=False)
    print('Created labelling file:', LABEL_PATH)

display(labeling[['row_id', 'headline', 'source_label', 'sentiment_label', 'label_reason', 'confidence']].head(15))

Existing labelling file loaded: C:\Users\brain\Documents\3mtt\3mtt_capstone_project\data\clean\articles_to_label.csv


,row_id,headline,source_label,sentiment_label,label_reason,confidence
0,1,Nigeria’s inflation rises to 15.69% in April a...,Premium Times Nigeria,negative,Inflation rising and food prices remain elevated,high
1,2,Food inflation rate continues on a steady rise...,Business News Nigeria,negative,Food inflation continues steady climb,high
2,3,Nigeria’s food inflation slows to 16.06% in Ap...,CNBC Africa,neutral,Deceleration only as prices still rise,medium
3,4,10 Cheapest Places to Live in Nigeria: The Ult...,nigeriahousingmarket.com,neutral,Informational guide with no clear valence,medium
4,5,Cost of Living in Lagos (2026 Practical Budget...,nigeriahousingmarket.com,neutral,Practical budget guide with no valence,medium
5,6,"Rising Food Prices Spark Calls For AgTech, Log...",Independent Newspaper Nigeria,negative,Rising food prices spark investment calls,medium
6,7,Nigeria’s Headline Inflation Rises to 15.69% i...,Proshare,negative,Inflation rises amid persistent food pressures,high
7,8,Nigeria's Inflation Edges Higher to 15.93% in ...,Proshare,negative,Inflation edges higher as pressures persist,high
8,9,"Cost of Healthy Diet in Nigeria Rises to N1,51...",Real Broadcasting Network,negative,Healthy diet cost rises as crisis deepens,high
9,10,Cost of living crisis reshapes Eid spending in...,Al Jazeera,negative,Crisis reshapes Eid spending downward,high


## Labelling task

Open `data/clean/articles_to_label.csv` in a spreadsheet or edit it through Colab. For every row, fill:

- `sentiment_label`: `positive`, `neutral`, or `negative`
- `label_reason`: one short explanation
- `confidence`: `high`, `medium`, or `low`


In [3]:
VALID_LABELS = {'positive', 'neutral', 'negative'}
VALID_CONFIDENCE = {'high', 'medium', 'low'}

labels = labeling['sentiment_label'].fillna('').str.strip().str.lower()
confidence = labeling['confidence'].fillna('').str.strip().str.lower()

print('Total rows:', len(labeling))
print('Unlabelled rows:', int(labels.eq('').sum()))
print('Rows needing review:', int(labels.eq('review').sum()))
print('Invalid labels:', int((~labels.isin(VALID_LABELS | {'' , 'review'})).sum()))
print('Label counts:')
display(labels.value_counts(dropna=False).to_frame('count'))
print('Invalid confidence values:', int((~confidence.isin(VALID_CONFIDENCE | {''})).sum()))

Total rows: 111
Unlabelled rows: 0
Rows needing review: 0
Invalid labels: 0
Label counts:


,count
sentiment_label,
negative,78
neutral,24
positive,9


Invalid confidence values: 0


In [4]:
complete = (
    labels.isin(VALID_LABELS).all()
    and confidence.isin(VALID_CONFIDENCE).all()
    and labeling['label_reason'].fillna('').str.strip().ne('').all()
)

if complete:
    labeling.to_csv(LABELED_PATH, index=False)
    print('All labels are complete. Saved:', LABELED_PATH)
else:
    print('Labelling is not complete. Resolve blank, review, invalid, or missing-reason rows first.')

All labels are complete. Saved: C:\Users\brain\Documents\3mtt\3mtt_capstone_project\data\clean\articles_labeled.csv


## Exit criteria

Day 4 labelling is complete only when all 111 rows have a final label, a reason, and a confidence value. We will then inspect class balance and begin the VADER and TF-IDF/Logistic Regression comparison.